In [6]:
import os
import json
import warnings
import psycopg2
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── Database connection ──────────────────────────────────────────────────────
conn = psycopg2.connect(
    host="127.0.0.1", port=5455, dbname="postgres",
    user="postgres", password="postgres"
)
print("DB connection established\n")

# ── Artifact paths ───────────────────────────────────────────────────────────
MAIN_ART  = "/Users/kevinjohnson/cfb-analytics/artifacts"
LOCAL_ART = "/Users/kevinjohnson/cfb-analytics/notebooks/artifacts"

verdict_files = {
    "epa":         os.path.join(MAIN_ART,  "epa_feature_verdict.csv"),
    "sp_rec":      os.path.join(MAIN_ART,  "sp_recruiting_verdict.csv"),
    "hierarchy":   os.path.join(MAIN_ART,  "hierarchy_verdict.csv"),   # CSV version
    "environment": os.path.join(MAIN_ART,  "environment_verdict.csv"),
    "momentum":    os.path.join(MAIN_ART,  "momentum_verdict.csv"),
    "elo":         os.path.join(MAIN_ART,  "elo_excitement_verdict.csv"),
    "style":       os.path.join(LOCAL_ART, "style_tempo_verdict.csv"),  # rebuilt Day 15
    "game_script": os.path.join(LOCAL_ART, "game_script_verdict.csv"),
}

missing = [k for k, v in verdict_files.items() if not os.path.exists(v)]
if missing:
    raise FileNotFoundError(f"Missing verdict files: {missing}")

print("All verdict files confirmed present:")
for k, v in verdict_files.items():
    print(f"  {k:15s} → {v}  ({os.path.getsize(v):,} bytes)")

# ── Load all verdict CSVs ────────────────────────────────────────────────────
verdicts = {k: pd.read_csv(v) for k, v in verdict_files.items()}

print("\n=== Row counts and columns per verdict file ===")
for k, df in verdicts.items():
    print(f"\n  [{k}]  {len(df)} rows")
    print(f"    columns: {list(df.columns)}")

# ── Load candidate_features for reference ───────────────────────────────────
candidate_features = pd.read_csv(os.path.join(MAIN_ART, "candidate_features.csv"))
keep_features = candidate_features[candidate_features["keep"] == True].copy()
print(f"\n=== candidate_features.csv ===")
print(f"  Total rows: {len(candidate_features)}")
print(f"  keep=True:  {len(keep_features)}")

DB connection established

All verdict files confirmed present:
  epa             → /Users/kevinjohnson/cfb-analytics/artifacts/epa_feature_verdict.csv  (1,131 bytes)
  sp_rec          → /Users/kevinjohnson/cfb-analytics/artifacts/sp_recruiting_verdict.csv  (2,997 bytes)
  hierarchy       → /Users/kevinjohnson/cfb-analytics/artifacts/hierarchy_verdict.csv  (805 bytes)
  environment     → /Users/kevinjohnson/cfb-analytics/artifacts/environment_verdict.csv  (3,249 bytes)
  momentum        → /Users/kevinjohnson/cfb-analytics/artifacts/momentum_verdict.csv  (2,069 bytes)
  elo             → /Users/kevinjohnson/cfb-analytics/artifacts/elo_excitement_verdict.csv  (1,328 bytes)
  style           → /Users/kevinjohnson/cfb-analytics/notebooks/artifacts/style_tempo_verdict.csv  (27,938 bytes)
  game_script     → /Users/kevinjohnson/cfb-analytics/notebooks/artifacts/game_script_verdict.csv  (1,387 bytes)

=== Row counts and columns per verdict file ===

  [epa]  5 rows
    columns: ['feature', 's

In [7]:
# ── Master verdict schema ────────────────────────────────────────────────────
MASTER_COLS = [
    "feature_name", "source_day", "spread_signal", "spread_partial_r",
    "spread_conferences", "ou_signal", "ou_partial_r", "ou_conferences",
    "moneyline_variance_signal", "yoy_stability_r", "yoy_verdict", "role",
    "ambiguity_flag", "final_decision", "decision_justification",
]

rows = []

def yn(val):
    if pd.isna(val):
        return "no"
    if isinstance(val, bool):
        return "yes" if val else "no"
    s = str(val).strip().lower()
    if s in ("yes", "true", "1", "signal", "confirmed", "holds"):
        return "yes"
    return "no"

def safe_r(val):
    try:
        v = float(val)
        return round(v, 4) if not np.isnan(v) else ""
    except (TypeError, ValueError):
        return ""

# ════════════════════════════════════════════════════════════════════════════
# DAY 8 — EPA
# ════════════════════════════════════════════════════════════════════════════
epa = verdicts["epa"]

epa_role_map = {
    "close_game_epa_per_play":     ("anchor",            "no",  "include", "Anchor pair confirmed: spread r=0.5988 at conf game 1, trajectory holds, YoY r=0.4331."),
    "close_game_def_epa_per_play": ("anchor",            "no",  "include", "Anchor pair confirmed: spread r=-0.6134 at conf game 1, trajectory holds, YoY r=0.4224."),
    "def_epa_per_play_allowed":    ("excluded",          "no",  "exclude", "Redundant: collinear with close_game_def_epa_per_play (r=0.9775)."),
    "last3_off_epa_avg":           ("conference_specific","no",  "include", "Conference-specific spread signal in ACC, Mid-American, SEC; null at conf game 1."),
    "last3_def_epa_avg":           ("conference_specific","no",  "include", "Conference-specific spread signal in American Athletic, Big Ten, Conference USA, Mid-American, Pac-12, Sun Belt; null at conf game 1."),
}

epa_conf_map = {
    "close_game_epa_per_play":     "all",
    "close_game_def_epa_per_play": "all",
    "last3_off_epa_avg":           "ACC,Mid-American,SEC",
    "last3_def_epa_avg":           "American Athletic,Big Ten,Conference USA,Mid-American,Pac-12,Sun Belt",
}

# Track features already added — Day 8 is authoritative for these
day8_features_added = set()

for _, r in epa.iterrows():
    fn = r["feature"]
    role, ambig, decision, justification = epa_role_map.get(
        fn, ("excluded", "no", "exclude", "Not in role map.")
    )
    rows.append({
        "feature_name":              fn,
        "source_day":                8,
        "spread_signal":             yn(r["spread_signal"]),
        "spread_partial_r":          safe_r(r["spread_partial_r"]),
        "spread_conferences":        epa_conf_map.get(fn, ""),
        "ou_signal":                 yn(r["ou_signal"]),
        "ou_partial_r":              safe_r(r["ou_partial_r"]),
        "ou_conferences":            "all" if yn(r["ou_signal"]) == "yes" else "",
        "moneyline_variance_signal": "no",
        "yoy_stability_r":           safe_r(r["yoy_r"]),
        "yoy_verdict":               "stable" if yn(r["yoy_stable"]) == "yes" else "not_applicable",
        "role":                      role,
        "ambiguity_flag":            ambig,
        "final_decision":            decision,
        "decision_justification":    justification,
    })
    day8_features_added.add(fn)

print(f"Day 8 EPA rows added: {len(epa)}")

# ════════════════════════════════════════════════════════════════════════════
# DAY 9 — SP+ and Recruiting
# sp_recruiting_verdict.csv has one row per conference breakdown.
# Deduplicate to one canonical row per feature using the role map.
# ════════════════════════════════════════════════════════════════════════════
sp = verdicts["sp_rec"]

sp_role_map = {
    "sp_rating": (
        "prior_seed", "include",
        "SP+ prior seed: YoY r=0.7740; prior remains relevant through games 9-12; do not aggressively down-weight.",
        0.7740,
    ),
    "sp_offense": (
        "excluded", "exclude",
        "Less stable than sp_rating (YoY r=0.6060); use sp_rating as anchor only.",
        0.6060,
    ),
    "sp_defense": (
        "excluded", "exclude",
        "Less stable than sp_rating (YoY r=0.6725); use sp_rating as anchor only.",
        0.6725,
    ),
    "recruiting_3yr_avg": (
        "prior_seed", "include",
        "Extremely stable prior seed (YoY r=0.9779); moderate weight in Big Ten and SEC; negative direction in Sun Belt.",
        0.9779,
    ),
    "opp_sp_rating_at_game_time": (
        "excluded", "exclude",
        "Control variable only; not a model feature or prior seed.",
        None,
    ),
}

# Emit exactly one row per feature — ignore extra conference breakdown rows
for fn, (role, decision, justification, yoy_r_val) in sp_role_map.items():
    rows.append({
        "feature_name":              fn,
        "source_day":                9,
        "spread_signal":             "no",
        "spread_partial_r":          "",
        "spread_conferences":        "",
        "ou_signal":                 "no",
        "ou_partial_r":              "",
        "ou_conferences":            "",
        "moneyline_variance_signal": "no",
        "yoy_stability_r":           yoy_r_val if yoy_r_val is not None else "",
        "yoy_verdict":               "stable" if role == "prior_seed" else "not_applicable",
        "role":                      role,
        "ambiguity_flag":            "no",
        "final_decision":            decision,
        "decision_justification":    justification,
    })

print(f"Day 9 SP/Rec rows added: {len(sp_role_map)}")

# ════════════════════════════════════════════════════════════════════════════
# DAY 10 — Hierarchy (structural decisions only, no feature rows)
# ════════════════════════════════════════════════════════════════════════════
print("Day 10 Hierarchy: structural decisions only — no feature rows added.")

# ════════════════════════════════════════════════════════════════════════════
# DAY 11 — Environmental Features
# ════════════════════════════════════════════════════════════════════════════
env = verdicts["environment"]

env_role_map = {
    "away_elevation_delta_ft":   ("anchor",    "no",  "include", "Threshold-activated anchor (>=2000ft): spread r=0.1518, YoY r=0.8255; signal concentrates in Mountain West and Big 12."),
    "venue_elevation_ft":        ("excluded",  "no",  "exclude", "Redundant: no threshold cleared; use away_elevation_delta_ft instead."),
    "away_travel_distance_mi":   ("supporting","no",  "include", "Threshold-activated supporting (>=1500mi): spread r=0.2011, YoY r=0.6562; spread signal only."),
    "away_tz_delta_hrs":         ("supporting","no",  "include", "Threshold-activated supporting (abs>=2hr): spread r=-0.2669, YoY r=0.6710; spread signal only."),
    "kickoff_hour":              ("excluded",  "no",  "exclude", "Insufficient sample when crossed with timezone (n=8); interaction not modeled."),
    "wind_speed_mph":            ("excluded",  "no",  "exclude", "Redundant: no signal after EPA control at any threshold; absorbed by EPA anchor pair."),
    "wind_gusts_mph":            ("excluded",  "no",  "exclude", "Redundant: no signal after EPA control at any threshold."),
    "is_high_wind":              ("excluded",  "no",  "exclude", "Redundant: no signal after EPA control at any threshold."),
    "wind_chill":                ("supporting","no",  "include", "O/U signal only at <=40F (r=0.1122), strengthens at <=25F (r=0.3373); no spread signal."),
    "temperature_f":             ("excluded",  "no",  "exclude", "Largely absorbed by wind_chill composite; redundant."),
    "humidity_pct":              ("excluded",  "no",  "exclude", "No signal in triggered population on clean 2022-2024 data."),
    "heat_index":                ("excluded",  "no",  "exclude", "O/U signal absent in triggered population on clean 2022-2024 data."),
    "precipitation_inches":      ("excluded",  "no",  "exclude", "Insufficient sample (n=44); do not model."),
    "is_precipitation":          ("excluded",  "no",  "exclude", "Insufficient sample (n=44); do not model."),
    "is_dome":                   ("excluded",  "no",  "exclude", "Redundant: dome override zeroes weather correctly; no residual signal after env controls."),
}

env_threshold_spread = {"away_elevation_delta_ft", "away_travel_distance_mi", "away_tz_delta_hrs"}

for _, r in env.iterrows():
    fn = r["feature"]
    if fn not in env_role_map:
        continue
    role, ambig, decision, justification = env_role_map[fn]
    spread_sig = "yes" if fn in env_threshold_spread else "no"
    ou_sig     = "yes" if fn == "wind_chill" else "no"
    rows.append({
        "feature_name":              fn,
        "source_day":                11,
        "spread_signal":             spread_sig,
        "spread_partial_r":          safe_r(r["spread_partial_r"]) if spread_sig == "yes" else "",
        "spread_conferences":        "Mountain West,Big 12" if fn == "away_elevation_delta_ft" else ("all" if spread_sig == "yes" else ""),
        "ou_signal":                 ou_sig,
        "ou_partial_r":              safe_r(r["ou_partial_r"]) if ou_sig == "yes" else "",
        "ou_conferences":            "all" if ou_sig == "yes" else "",
        "moneyline_variance_signal": "no",
        "yoy_stability_r":           safe_r(r["yoy_r"]),
        "yoy_verdict":               "stable" if yn(r.get("yoy_stable", "no")) == "yes" else ("unstable" if safe_r(r["yoy_r"]) != "" else "not_applicable"),
        "role":                      role,
        "ambiguity_flag":            ambig,
        "final_decision":            decision,
        "decision_justification":    justification,
    })

print(f"Day 11 Environment rows added: {len(env_role_map)}")

# ════════════════════════════════════════════════════════════════════════════
# DAY 12 — Momentum / Rolling Features
# Skip any feature already entered from Day 8 — Day 8 is authoritative.
# ════════════════════════════════════════════════════════════════════════════
mom = verdicts["momentum"]

mom_conf_map = {
    "last3_off_epa_avg":        "ACC,Conference USA,Mid-American,Mountain West,SEC",
    "last3_def_epa_avg":        "American Athletic,Big Ten,Conference USA,Mid-American,Pac-12,Sun Belt",
    "last3_points_scored_avg":  "ACC,Big 12,Big Ten,Conference USA,Mid-American,Mountain West",
    "last3_points_allowed_avg": "American Athletic,Big Ten,Conference USA,Mountain West,Pac-12,Sun Belt",
    "last3_win_pct":            "all",
    "days_since_last_game":     "American Athletic,Big 12",
}

mom_role_map = {
    "last3_off_epa_avg":        ("conference_specific", "no",  "include", "Conference-specific spread signal in 5 conferences; null at conf game 1. Day 8 is authoritative verdict source."),
    "last3_def_epa_avg":        ("conference_specific", "no",  "include", "Conference-specific spread signal in 6 conferences; null at conf game 1. Day 8 is authoritative verdict source."),
    "last3_points_scored_avg":  ("conference_specific", "no",  "include", "Conference-specific supporting; signal holds from conf game 2 in 6 conferences."),
    "last3_points_allowed_avg": ("conference_specific", "no",  "include", "Conference-specific supporting; signal holds from conf game 2 in 6 conferences."),
    "last3_win_pct":            ("supporting",          "no",  "include", "Broad supporting spread signal from conf game 2 across conferences."),
    "days_since_last_game":     ("conference_specific", "no",  "include", "Bye week signal (>=12d) in American Athletic and Big 12 only; redundant elsewhere."),
}

for _, r in mom.iterrows():
    fn = r["feature"]
    if fn not in mom_role_map:
        continue
    # Skip if Day 8 already added this feature — Day 8 verdict is authoritative
    if fn in day8_features_added:
        continue
    role, ambig, decision, justification = mom_role_map[fn]
    rows.append({
        "feature_name":              fn,
        "source_day":                12,
        "spread_signal":             yn(r["spread_signal"]),
        "spread_partial_r":          safe_r(r["spread_partial_r"]),
        "spread_conferences":        mom_conf_map.get(fn, ""),
        "ou_signal":                 yn(r["ou_signal"]),
        "ou_partial_r":              safe_r(r["ou_partial_r"]),
        "ou_conferences":            "",
        "moneyline_variance_signal": yn(r.get("variance_signal_spread", "no")),
        "yoy_stability_r":           safe_r(r["yoy_r"]),
        "yoy_verdict":               "stable" if yn(r.get("yoy_stable", "no")) == "yes" else "unstable",
        "role":                      role,
        "ambiguity_flag":            ambig,
        "final_decision":            decision,
        "decision_justification":    justification,
    })

print(f"Day 12 Momentum rows added (after Day 8 dedup): {sum(1 for r in rows if r['source_day'] == 12)}")

# ════════════════════════════════════════════════════════════════════════════
# DAY 13 — ELO and Excitement
# ════════════════════════════════════════════════════════════════════════════
elo = verdicts["elo"]

elo_role_map = {
    "pregame_elo": (
        "supporting", "no", "include",
        "Game-level spread predictor: r=0.1702 full population, holds at conf game 1 (r=0.1870); YoY r=0.8452.",
    ),
    "elo_sp_divergence": (
        "supporting", "yes", "include",
        "Spread r=0.1650 after SP+ controlled; compute in notebook first, add to dbt after model confirms value. (Ambiguity 5 resolved: include.)",
    ),
    "prior_avg_excitement_index": (
        "excluded", "no", "exclude",
        "YoY r=0.1877 — extremely unstable; late-season O/U signal does not hold earlier; unusable as prior seed.",
    ),
}

for _, r in elo.iterrows():
    fn = r["feature"]
    if fn not in elo_role_map:
        continue
    role, ambig, decision, justification = elo_role_map[fn]
    rows.append({
        "feature_name":              fn,
        "source_day":                13,
        "spread_signal":             yn(r["spread_signal"]),
        "spread_partial_r":          safe_r(r["spread_partial_r"]),
        "spread_conferences":        "all" if yn(r["spread_signal"]) == "yes" else "",
        "ou_signal":                 yn(r["ou_signal"]),
        "ou_partial_r":              safe_r(r["ou_partial_r"]),
        "ou_conferences":            "",
        "moneyline_variance_signal": "no",
        "yoy_stability_r":           safe_r(r["yoy_r"]),
        "yoy_verdict":               "stable" if yn(r.get("yoy_stable", "no")) == "yes" else "unstable",
        "role":                      role,
        "ambiguity_flag":            ambig,
        "final_decision":            decision,
        "decision_justification":    justification,
    })

print(f"Day 13 ELO rows added: {len(elo_role_map)}")

# ════════════════════════════════════════════════════════════════════════════
# DAY 14 — raw.plays-derived candidates (no verdict CSV; encoded from session state)
# ════════════════════════════════════════════════════════════════════════════
day14_features = [
    ("success_rate_overall",            "No EDA signal test run; excluded pending model-phase evaluation."),
    ("success_rate_rush",               "No EDA signal test run; excluded pending model-phase evaluation."),
    ("success_rate_pass",               "No EDA signal test run; excluded pending model-phase evaluation."),
    ("success_rate_std_downs",          "No EDA signal test run; excluded pending model-phase evaluation."),
    ("success_rate_pass_downs",         "No EDA signal test run; excluded pending model-phase evaluation."),
    ("stuff_rate",                      "No EDA signal test run; excluded pending model-phase evaluation."),
    ("explosive_rate_20",               "No EDA signal test run; excluded pending model-phase evaluation."),
    ("explosive_rate_10",               "No EDA signal test run; excluded pending model-phase evaluation."),
    ("line_yards_per_rush",             "No EDA signal test run; excluded pending model-phase evaluation."),
    ("off_sack_rate_allowed",           "No EDA signal test run; excluded pending model-phase evaluation."),
    ("def_sack_rate",                   "No EDA signal test run; excluded pending model-phase evaluation."),
    ("off_pts_per_opportunity",         "No EDA signal test run; excluded pending model-phase evaluation."),
    ("def_pts_per_opportunity_allowed", "No EDA signal test run; excluded pending model-phase evaluation."),
    ("off_epa_rush",                    "No EDA signal test run; excluded pending model-phase evaluation."),
    ("off_epa_pass",                    "No EDA signal test run; excluded pending model-phase evaluation."),
    ("def_epa_rush_allowed",            "No EDA signal test run; excluded pending model-phase evaluation."),
    ("def_epa_pass_allowed",            "No EDA signal test run; excluded pending model-phase evaluation."),
    ("time_of_possession_seconds",      "No EDA signal test run; excluded pending model-phase evaluation."),
    ("redzone_success_rate",            "No EDA signal test run; excluded pending model-phase evaluation."),
    ("redzone_epa",                     "No EDA signal test run; excluded pending model-phase evaluation."),
    ("redzone_pts_per_opportunity",     "No EDA signal test run; excluded pending model-phase evaluation."),
    ("midfield_success_rate",           "No EDA signal test run; excluded pending model-phase evaluation."),
    ("midfield_epa",                    "No EDA signal test run; excluded pending model-phase evaluation."),
    ("own_territory_success_rate",      "No EDA signal test run; excluded pending model-phase evaluation."),
    ("own_territory_epa",               "No EDA signal test run; excluded pending model-phase evaluation."),
    ("rush_rate_overall",               "No EDA signal test run; excluded pending model-phase evaluation."),
    ("rush_rate_std_downs_season",      "In-game version tested in Day 15; season-level version not yet tested; excluded pending model-phase evaluation."),
    ("rush_rate_pass_downs_season",     "In-game version tested in Day 15; season-level version not yet tested; excluded pending model-phase evaluation."),
    ("avg_drive_epa",                   "No EDA signal test run; excluded pending model-phase evaluation."),
    ("avg_drive_success_rate",          "No EDA signal test run; excluded pending model-phase evaluation."),
    ("scoring_drive_rate",              "No EDA signal test run; excluded pending model-phase evaluation."),
]

for fn, justification in day14_features:
    rows.append({
        "feature_name":              fn,
        "source_day":                14,
        "spread_signal":             "no",
        "spread_partial_r":          "",
        "spread_conferences":        "",
        "ou_signal":                 "no",
        "ou_partial_r":              "",
        "ou_conferences":            "",
        "moneyline_variance_signal": "no",
        "yoy_stability_r":           "",
        "yoy_verdict":               "not_applicable",
        "role":                      "supporting",
        "ambiguity_flag":            "no",
        "final_decision":            "exclude",
        "decision_justification":    justification,
    })

print(f"Day 14 raw.plays candidates added: {len(day14_features)}")

# ════════════════════════════════════════════════════════════════════════════
# DAY 15 — Style / Tempo Delta
# Weak prior-seed candidates (rush_rate_std_downs, rush_rate_pass_downs) are
# assigned role=supporting here. Prior-seed designation resolved in ambiguity
# resolution and carried to final_features only if included.
# ════════════════════════════════════════════════════════════════════════════
style = verdicts["style"]

style_ambig_features = {"rush_rate_std_downs", "rush_rate_pass_downs"}

def map_style_role(row):
    v = str(row.get("final_verdict", "")).lower()
    r = str(row.get("recommended_deployment_role", "")).lower()
    if "exclude" in v or "exclude" in r:
        return "excluded"
    if "anchor" in r:
        return "anchor"
    if "conference" in r:
        return "conference_specific"
    # Treat weak prior-seed candidates as supporting in master_verdict
    if "supporting" in r or "include" in v or "prior" in r:
        return "supporting"
    return "excluded"

for _, r in style.iterrows():
    fn = r["feature_delta"] if pd.notna(r.get("feature_delta")) else r["feature"]
    spread_sig = yn(r.get("spread_verdict", "no"))
    ou_sig     = yn(r.get("over_under_verdict", "no"))
    ml_sig     = yn(r.get("moneyline_variance_verdict", "no"))
    yoy_r      = safe_r(r.get("avg_yoy_r", ""))
    yoy_v      = "stable" if yn(r.get("stable_for_prior_seed", "no")) == "yes" else "unstable"
    role       = map_style_role(r)
    ambig      = "yes" if fn in style_ambig_features else "no"
    decision   = "exclude" if role == "excluded" else "include"

    sc = str(r.get("spread_conferences_clearing", "")).strip()
    sc = "" if sc in ("nan", "") else sc

    oc = str(r.get("ou_conferences_clearing", "")).strip()
    oc = "" if oc in ("nan", "") else oc

    note = str(r.get("methodology_note", "")).strip()
    if not note or note == "nan":
        note = f"Style delta: spread={spread_sig}, O/U={ou_sig}, ML variance={ml_sig}, YoY r={yoy_r}."

    rows.append({
        "feature_name":              fn,
        "source_day":                15,
        "spread_signal":             spread_sig,
        "spread_partial_r":          safe_r(r.get("spread_full_partial_r", "")),
        "spread_conferences":        sc,
        "ou_signal":                 ou_sig,
        "ou_partial_r":              safe_r(r.get("ou_full_partial_r", "")),
        "ou_conferences":            oc,
        "moneyline_variance_signal": ml_sig,
        "yoy_stability_r":           yoy_r,
        "yoy_verdict":               yoy_v,
        "role":                      role,
        "ambiguity_flag":            ambig,
        "final_decision":            decision,
        "decision_justification":    note,
    })

print(f"Day 15 Style/Tempo rows added: {len(style)}")

# ════════════════════════════════════════════════════════════════════════════
# DAY 16 — Style Archetypes (no verdict CSV; encoded from session state)
# Ambiguity 2 resolved: include as game-level O/U features with deployable
# pregame version requirement.
# ════════════════════════════════════════════════════════════════════════════
day16_features = [
    ("offense_archetype_matchup",    "conference_specific", "yes", "yes", "no",
     "O/U confirmed (eta2=0.3684), weak spread secondary (eta2=0.0188); not valid for ML variance; not stable for prior seeding; deployable pregame version required. (Ambiguity 2 resolved: include.)"),
    ("defense_archetype_matchup",    "conference_specific", "yes", "yes", "no",
     "O/U confirmed (eta2=0.3901), weak spread secondary (eta2=0.0213); not valid for ML variance; not stable for prior seeding; deployable pregame version required. (Ambiguity 2 resolved: include.)"),
    ("home_off_vs_away_def_matchup", "supporting",          "yes", "yes", "no",
     "O/U confirmed (eta2=0.2234), weak spread secondary (eta2=0.0103); deployable pregame version required. (Ambiguity 2 resolved: include.)"),
    ("away_off_vs_home_def_matchup", "supporting",          "no",  "yes", "no",
     "O/U confirmed (eta2=0.2328); spread secondary did not clear threshold; deployable pregame version required. (Ambiguity 2 resolved: include.)"),
]

for fn, role, spread_sig, ou_sig, ml_sig, justification in day16_features:
    rows.append({
        "feature_name":              fn,
        "source_day":                16,
        "spread_signal":             spread_sig,
        "spread_partial_r":          "",
        "spread_conferences":        "",
        "ou_signal":                 ou_sig,
        "ou_partial_r":              "",
        "ou_conferences":            "all",
        "moneyline_variance_signal": ml_sig,
        "yoy_stability_r":           "",
        "yoy_verdict":               "unstable",
        "role":                      role,
        "ambiguity_flag":            "yes",
        "final_decision":            "include",
        "decision_justification":    justification,
    })

print(f"Day 16 Archetype rows added: {len(day16_features)}")

# ════════════════════════════════════════════════════════════════════════════
# DAY 17 — Game Script
# Ambiguity 1 resolved: close_game_play_count_delta included for 6 confirmed
# conferences.
# ════════════════════════════════════════════════════════════════════════════
gs = verdicts["game_script"]

gs_role_map = {
    "game_script_avg_margin_delta": (
        "excluded", "no", "exclude",
        "Diagnostic only: near-tautological with point_differential (raw r=0.9075); post-game feature.",
    ),
    "game_script_ordinal_delta": (
        "excluded", "no", "exclude",
        "Diagnostic only: derived from in-game score margin (raw r=0.8628); post-game feature.",
    ),
    "close_game_play_count_delta": (
        "conference_specific", "yes", "include",
        "Conference-specific spread signal in 6/10 conferences (ACC, American Athletic, Big 12, Mid-American, Pac-12, Sun Belt); holds from game 1; no O/U or ML variance signal. (Ambiguity 1 resolved: include.)",
    ),
}

for _, r in gs.iterrows():
    fn = r["feature"]
    if fn not in gs_role_map:
        continue
    role, ambig, decision, justification = gs_role_map[fn]
    rows.append({
        "feature_name":              fn,
        "source_day":                17,
        "spread_signal":             yn(r["spread_signal"]),
        "spread_partial_r":          "",
        "spread_conferences":        "ACC,American Athletic,Big 12,Mid-American,Pac-12,Sun Belt" if fn == "close_game_play_count_delta" else "",
        "ou_signal":                 yn(r["ou_signal"]),
        "ou_partial_r":              "",
        "ou_conferences":            "",
        "moneyline_variance_signal": yn(r["ml_variance"]),
        "yoy_stability_r":           "",
        "yoy_verdict":               "not_applicable",
        "role":                      role,
        "ambiguity_flag":            ambig,
        "final_decision":            decision,
        "decision_justification":    justification,
    })

print(f"Day 17 Game Script rows added: {len(gs_role_map)}")

# ════════════════════════════════════════════════════════════════════════════
# Assemble and verify
# ════════════════════════════════════════════════════════════════════════════
master = pd.DataFrame(rows, columns=MASTER_COLS)

# Confirm no duplicate feature_names
dupes = master[master.duplicated("feature_name", keep=False)]
if len(dupes) > 0:
    print(f"\n⚠ DUPLICATE FEATURE NAMES ({len(dupes)} rows):")
    print(dupes[["feature_name", "source_day"]].to_string())
else:
    print("\n✓ No duplicate feature names")

print(f"\n=== master_verdict assembled ===")
print(f"  Total rows:          {len(master)}")
print(f"\n  final_decision breakdown:")
print(master["final_decision"].value_counts().to_string())
print(f"\n  role breakdown:")
print(master["role"].value_counts().to_string())
print(f"\n  ambiguity_flag=yes rows:")
print(master[master["ambiguity_flag"] == "yes"][["feature_name", "source_day", "role", "final_decision"]].to_string())
print(f"\n  source_day counts:")
print(master["source_day"].value_counts().sort_index().to_string())

Day 8 EPA rows added: 5
Day 9 SP/Rec rows added: 5
Day 10 Hierarchy: structural decisions only — no feature rows added.
Day 11 Environment rows added: 15
Day 12 Momentum rows added (after Day 8 dedup): 4
Day 13 ELO rows added: 3
Day 14 raw.plays candidates added: 31
Day 15 Style/Tempo rows added: 24
Day 16 Archetype rows added: 4
Day 17 Game Script rows added: 3

✓ No duplicate feature names

=== master_verdict assembled ===
  Total rows:          93

  final_decision breakdown:
final_decision
exclude    70
include    23

  role breakdown:
role
supporting             41
excluded               39
conference_specific     8
anchor                  3
prior_seed              2

  ambiguity_flag=yes rows:
                    feature_name  source_day                 role final_decision
29             elo_sp_divergence          13           supporting        include
86     offense_archetype_matchup          16  conference_specific        include
87     defense_archetype_matchup          16  co

In [8]:
# ── Fix ambiguity flags missed in assembly ───────────────────────────────────
# Ambiguity 3: last3_off_epa_avg and last3_def_epa_avg
#   Conference lists differ between Day 8 EPA and Day 12 momentum verdicts.
#   Resolution: include with explicit conference lists; let conference-level
#   pooling handle variation within those lists. Do not consolidate into a
#   single rolling EPA feature.
ambiguity3_features = {"last3_off_epa_avg", "last3_def_epa_avg"}
master.loc[master["feature_name"].isin(ambiguity3_features), "ambiguity_flag"] = "yes"

# Ambiguity 4: rush_rate_std_downs and rush_rate_pass_downs as prior seeds
#   YoY r = 0.4890 and 0.4648 — below 0.5 threshold for prior seed.
#   Resolution: EXCLUDE as prior seeds. YoY stability insufficient.
#   They remain included as game-level supporting features (spread signal
#   confirmed in Day 15), but do not seed priors.
ambiguity4_features = {"rush_rate_std_downs_delta", "rush_rate_pass_downs_delta"}
master.loc[master["feature_name"].isin(ambiguity4_features), "ambiguity_flag"] = "yes"

print("Ambiguity flag counts after fix:")
print(master["ambiguity_flag"].value_counts().to_string())
print(f"\nAll ambiguity_flag=yes rows:")
print(master[master["ambiguity_flag"] == "yes"][
    ["feature_name", "source_day", "role", "final_decision"]
].to_string())

# ── Write master_verdict.csv ─────────────────────────────────────────────────
master_path = os.path.join(MAIN_ART, "master_verdict.csv")
master.to_csv(master_path, index=False)
print(f"\n✓ master_verdict.csv written → {master_path}  ({os.path.getsize(master_path):,} bytes)")
print(f"  Rows: {len(master)}  |  include: {(master['final_decision']=='include').sum()}  |  exclude: {(master['final_decision']=='exclude').sum()}")

# ── Write ambiguity_resolution.md ───────────────────────────────────────────
ambiguity_md = """\
# Ambiguity Resolution — Day 19

Five features finished EDA with open questions. Each receives a binding binary
decision below. These decisions are encoded in master_verdict.csv and
final_features.csv. They are final — no further deliberation in model build.

---

## Ambiguity 1 — close_game_play_count_delta

**Question:** Include as conference-specific spread feature for the 6 confirmed
conferences, or exclude on complexity grounds?

**Decision: INCLUDE**

**Justification:** Spread partial r=0.1834 full population (p<0.0001); holds from
conf game 1 (r=0.1676); confirmed in ACC, American Athletic, Big 12, Mid-American,
Pac-12, Sun Belt. Signal is not tautological (raw r=0.2256). The conference scope
is explicit and the feature is available at all information states. Complexity cost
is low — one additional feature with a known conference allowlist.

**Scope:** ACC, American Athletic, Big 12, Mid-American, Pac-12, Sun Belt only.
Not applied in Big Ten, Conference USA, Mountain West, or SEC.

---

## Ambiguity 2 — Style archetype matchup features

**Question:** Include as game-level O/U features with a deployable pregame version
requirement, or exclude until a pregame version is tested?

**Decision: INCLUDE** (with condition)

**Justification:** O/U signal is the strongest EDA 10 finding — eta² = 0.39 for
defense matchup, 0.37 for offense matchup. Signal held broadly across tiers,
seasons, and conferences. The in-game version cannot be used in production directly,
but the signal magnitude justifies retaining the features with an explicit condition:
a deployable pregame or rolling version must be tested before the model goes live.
If no pregame version clears signal tests, these features are dropped at that stage.

**Condition:** Deployable pregame version required before September 24, 2026
production launch. Features flagged for pregame version development in model build.

**Scope:** All four matchup features included (offense_archetype_matchup,
defense_archetype_matchup, home_off_vs_away_def_matchup,
away_off_vs_home_def_matchup). O/U signal only. Not used for spread or
moneyline variance.

---

## Ambiguity 3 — last3_off_epa_avg and last3_def_epa_avg conference lists

**Question:** Include as conference-specific features with explicit conference
lists, or consolidate into a single rolling EPA feature and let conference-level
pooling handle the variation?

**Decision: INCLUDE with explicit conference lists**

**Justification:** The conference-specific signal is real and the lists are
different for offense and defense — consolidating would lose that structure.
Conference-level pooling regularizes within-conference variation but does not
substitute for the binary include/exclude signal difference between conferences.
The explicit lists are: last3_off_epa_avg → ACC, Mid-American, SEC;
last3_def_epa_avg → American Athletic, Big Ten, Conference USA, Mid-American,
Pac-12, Sun Belt. Both are null at conf game 1 and handled by null_handling =
impute_season_prior.

---

## Ambiguity 4 — rush_rate_std_downs and rush_rate_pass_downs as prior seeds

**Question:** Include as weak prior seeds (YoY r = 0.4890 and 0.4648) or exclude
on grounds that YoY stability below 0.5 is insufficient for a prior seed role?

**Decision: EXCLUDE as prior seeds**

**Justification:** YoY r below 0.5 is insufficient to seed a prior — the signal
is too unstable to anchor a parameter before any in-season data arrives. Both
features are retained as game-level supporting predictors (spread signal confirmed
in Day 15), but they do not receive prior_seed role and do not contribute to prior
specification. Their in-game signal is captured through the game-level feature
pathway, not through prior seeding.

---

## Ambiguity 5 — elo_sp_divergence

**Question:** Include as game-level predictor computed in notebook, or exclude
until model confirms value?

**Decision: INCLUDE** (computed in notebook)

**Justification:** Spread r=0.1650 after SP+ controlled — ELO adds signal beyond
SP+ for spread prediction. The compute-in-notebook-first constraint is already a
locked decision. Including it here means Day 20 writes a prior for it and the model
build tests it. If it fails to improve the model, it is dropped in refinement
(Day 31). Excluding it now would mean losing a confirmed signal without testing it
in the model. The cost of including a weak feature is lower than the cost of
prematurely excluding a confirmed one.
"""

ambiguity_path = os.path.join(MAIN_ART, "ambiguity_resolution.md")
with open(ambiguity_path, "w") as f:
    f.write(ambiguity_md)
print(f"\n✓ ambiguity_resolution.md written → {ambiguity_path}  ({os.path.getsize(ambiguity_path):,} bytes)")

Ambiguity flag counts after fix:
ambiguity_flag
no     83
yes    10

All ambiguity_flag=yes rows:
                    feature_name  source_day                 role final_decision
3              last3_off_epa_avg           8  conference_specific        include
4              last3_def_epa_avg           8  conference_specific        include
29             elo_sp_divergence          13           supporting        include
73     rush_rate_std_downs_delta          15           supporting        include
74    rush_rate_pass_downs_delta          15           supporting        include
86     offense_archetype_matchup          16  conference_specific        include
87     defense_archetype_matchup          16  conference_specific        include
88  home_off_vs_away_def_matchup          16           supporting        include
89  away_off_vs_home_def_matchup          16           supporting        include
92   close_game_play_count_delta          17  conference_specific        include

✓ master_v

In [9]:
# ── Start from included features only ───────────────────────────────────────
final = master[master["final_decision"] == "include"].copy().reset_index(drop=True)
print(f"Starting with {len(final)} included features")

# ── Add required final_features columns ─────────────────────────────────────
# Populated by explicit lookup — every feature must have a complete entry.
# Columns: model_layer, prior_type, prior_mean, prior_sd, prior_justification,
#          conference_scope, threshold_spec, null_handling

PRIOR_SPECS = {
    # ── ANCHORS ─────────────────────────────────────────────────────────────
    "close_game_epa_per_play": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.5,
        "prior_justification": "Game-level anchor; weakly informative normal centered at zero; scale informed by partial r=0.5988 and YoY r=0.4331.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "close_game_def_epa_per_play": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.5,
        "prior_justification": "Game-level anchor; weakly informative normal centered at zero; scale informed by partial r=-0.6134 and YoY r=0.4224.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "away_elevation_delta_ft": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Threshold-activated anchor (>=2000ft); YoY r=0.8255; weakly informative normal; effect expected negative for visiting team.",
        "conference_scope":  "Mountain West,Big 12",
        "threshold_spec":    "away_elevation_delta_ft >= 2000",
        "null_handling":     "zero",
    },

    # ── PRIOR SEEDS ─────────────────────────────────────────────────────────
    "sp_rating": {
        "model_layer":       "team",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          1.0,
        "prior_justification": "Prior seed: YoY r=0.7740; informative prior centered at SP+ preseason rating; does not decay monotonically — weight maintained through games 9-12.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "impute_season_prior",
    },
    "recruiting_3yr_avg": {
        "model_layer":       "team",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.5,
        "prior_justification": "Prior seed: YoY r=0.9779; informative prior; moderate weight in Big Ten and SEC; low weight elsewhere; non-positive direction in Sun Belt (do not use as positive signal).",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "impute_season_prior",
    },

    # ── SUPPORTING — ENVIRONMENTAL ───────────────────────────────────────────
    "away_travel_distance_mi": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Threshold-activated supporting (>=1500mi); spread r=0.2011; YoY r=0.6562; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "away_travel_distance_mi >= 1500",
        "null_handling":     "zero",
    },
    "away_tz_delta_hrs": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Threshold-activated supporting (abs>=2hr); spread r=-0.2669; YoY r=0.6710; weakly informative normal; direction negative.",
        "conference_scope":  "all",
        "threshold_spec":    "abs(away_tz_delta_hrs) >= 2",
        "null_handling":     "zero",
    },
    "wind_chill": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.2,
        "prior_justification": "O/U supporting only; triggered at <=40F (r=0.1122), strengthens at <=25F (r=0.3373); weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "wind_chill <= 40 AND NOT is_dome",
        "null_handling":     "zero",
    },

    # ── SUPPORTING — ELO ────────────────────────────────────────────────────
    "pregame_elo": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Game-level spread predictor; partial r=0.1702; holds at conf game 1; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "elo_sp_divergence": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.2,
        "prior_justification": "Spread r=0.1650 after SP+ controlled; weakly informative normal; computed in notebook; add to dbt only after model confirms value.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },

    # ── SUPPORTING — MOMENTUM ───────────────────────────────────────────────
    "last3_win_pct": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Broad supporting spread signal from conf game 2; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "impute_season_prior",
    },

    # ── CONFERENCE-SPECIFIC — EPA ROLLING ───────────────────────────────────
    "last3_off_epa_avg": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Conference-specific spread signal (ACC, Mid-American, SEC); null at conf game 1; weakly informative normal.",
        "conference_scope":  "ACC,Mid-American,SEC",
        "threshold_spec":    "",
        "null_handling":     "impute_season_prior",
    },
    "last3_def_epa_avg": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Conference-specific spread signal (American Athletic, Big Ten, Conference USA, Mid-American, Pac-12, Sun Belt); null at conf game 1; weakly informative normal.",
        "conference_scope":  "American Athletic,Big Ten,Conference USA,Mid-American,Pac-12,Sun Belt",
        "threshold_spec":    "",
        "null_handling":     "impute_season_prior",
    },

    # ── CONFERENCE-SPECIFIC — MOMENTUM ──────────────────────────────────────
    "last3_points_scored_avg": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Conference-specific supporting; signal from conf game 2 in 6 conferences; weakly informative normal.",
        "conference_scope":  "ACC,Big 12,Big Ten,Conference USA,Mid-American,Mountain West",
        "threshold_spec":    "",
        "null_handling":     "impute_season_prior",
    },
    "last3_points_allowed_avg": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Conference-specific supporting; signal from conf game 2 in 6 conferences; weakly informative normal.",
        "conference_scope":  "American Athletic,Big Ten,Conference USA,Mountain West,Pac-12,Sun Belt",
        "threshold_spec":    "",
        "null_handling":     "impute_season_prior",
    },
    "days_since_last_game": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.2,
        "prior_justification": "Bye week signal (>=12d) in American Athletic and Big 12 only; weakly informative normal.",
        "conference_scope":  "American Athletic,Big 12",
        "threshold_spec":    "days_since_last_game >= 12",
        "null_handling":     "zero",
    },

    # ── CONFERENCE-SPECIFIC — GAME SCRIPT ───────────────────────────────────
    "close_game_play_count_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.2,
        "prior_justification": "Conference-specific spread signal in 6 conferences; holds from game 1; weakly informative normal. (Ambiguity 1 resolved: include.)",
        "conference_scope":  "ACC,American Athletic,Big 12,Mid-American,Pac-12,Sun Belt",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },

    # ── STYLE / TEMPO DELTA — SPREAD SUPPORTING ──────────────────────────────
    "rush_rate_std_downs_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Spread signal confirmed across all season buckets including game 1 (r=0.2965–0.3628); weakly informative normal; not a prior seed (YoY r=0.4890 insufficient).",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "rush_rate_pass_downs_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Spread signal confirmed; weakly informative normal; not a prior seed (YoY r=0.4648 insufficient).",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "off_pts_per_opportunity_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Spread signal confirmed in Day 15; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "def_pts_per_opportunity_allowed_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Spread signal confirmed in Day 15; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "off_success_rate_pass_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Spread signal confirmed in Day 15; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "def_success_rate_pass_allowed_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Spread signal confirmed in Day 15; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "off_epa_pass_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Spread signal confirmed in Day 15; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "def_epa_pass_allowed_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "Spread signal confirmed in Day 15; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },

    # ── STYLE / TEMPO DELTA — MONEYLINE VARIANCE ────────────────────────────
    "off_sack_rate_allowed_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.2,
        "prior_justification": "Moneyline variance candidate; abs residual partial r=+0.0919; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "def_sack_rate_delta": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.2,
        "prior_justification": "Moneyline variance candidate; abs residual partial r=-0.0919; weakly informative normal.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },

    # ── ARCHETYPES ───────────────────────────────────────────────────────────
    "offense_archetype_matchup": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "O/U feature (eta2=0.3684); weakly informative normal; deployable pregame version required before production. (Ambiguity 2 resolved: include.)",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "defense_archetype_matchup": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "O/U feature (eta2=0.3901); weakly informative normal; deployable pregame version required before production. (Ambiguity 2 resolved: include.)",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "home_off_vs_away_def_matchup": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "O/U feature (eta2=0.2234); weakly informative normal; deployable pregame version required.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
    "away_off_vs_home_def_matchup": {
        "model_layer":       "game_level",
        "prior_type":        "normal",
        "prior_mean":        0.0,
        "prior_sd":          0.3,
        "prior_justification": "O/U feature (eta2=0.2328); weakly informative normal; deployable pregame version required.",
        "conference_scope":  "all",
        "threshold_spec":    "",
        "null_handling":     "not_applicable",
    },
}

# ── Apply prior specs to final_features ─────────────────────────────────────
new_cols = ["model_layer", "prior_type", "prior_mean", "prior_sd",
            "prior_justification", "conference_scope", "threshold_spec", "null_handling"]

for col in new_cols:
    final[col] = ""

missing_specs = []
for idx, row in final.iterrows():
    fn = row["feature_name"]
    if fn in PRIOR_SPECS:
        for col in new_cols:
            final.at[idx, col] = PRIOR_SPECS[fn][col]
    else:
        missing_specs.append(fn)

if missing_specs:
    print(f"\n⚠ Features with no prior spec ({len(missing_specs)}) — will be EXCLUDED from final_features:")
    for fn in missing_specs:
        print(f"  {fn}")
    # Drop features with no prior spec — per spec: cannot enter final_features without complete prior
    final = final[~final["feature_name"].isin(missing_specs)].reset_index(drop=True)
else:
    print("✓ All included features have prior specifications")

# ── Write final_features.csv ─────────────────────────────────────────────────
final_cols = MASTER_COLS + new_cols
final_path = os.path.join(MAIN_ART, "final_features.csv")
final[final_cols].to_csv(final_path, index=False)

print(f"\n=== final_features.csv ===")
print(f"  Rows: {len(final)}")
print(f"\n  Role breakdown:")
print(final["role"].value_counts().to_string())
print(f"\n  Model layer breakdown:")
print(final["model_layer"].value_counts().to_string())
print(f"\n  Null handling breakdown:")
print(final["null_handling"].value_counts().to_string())
print(f"\n✓ final_features.csv written → {final_path}  ({os.path.getsize(final_path):,} bytes)")

Starting with 23 included features
✓ All included features have prior specifications

=== final_features.csv ===
  Rows: 23

  Role breakdown:
role
supporting             10
conference_specific     8
anchor                  3
prior_seed              2

  Model layer breakdown:
model_layer
game_level    21
team           2

  Null handling breakdown:
null_handling
not_applicable         11
impute_season_prior     7
zero                    5

✓ final_features.csv written → /Users/kevinjohnson/cfb-analytics/artifacts/final_features.csv  (10,007 bytes)


In [10]:
prior_spec_md = """\
# Prior Specification Draft — Day 19

This document specifies every prior distribution the hierarchical Negative Binomial
model will use. It is not PyMC code. Day 20 (model_01_prior_specification.ipynb)
translates this into PyMC code. Every prior is traceable to a YoY r value, partial r,
or named EDA finding. No prior may be invented in Day 20 without a corresponding
entry here.

---

## 1. League-Level Priors

### 1.1 Intercept (league baseline scoring)
- **Parameter:** mu_league
- **Distribution:** Normal
- **Mean:** 27.0
- **SD:** 5.0
- **Type:** Weakly informative
- **Justification:** FBS mean points scored per team per game 2022–2024 is
  approximately 27. SD of 5 allows the posterior to move freely while ruling
  out implausible baselines (e.g. 0 or 60 points).

### 1.2 Home field advantage baseline
- **Parameter:** hfa_league
- **Distribution:** Normal
- **Mean:** 2.5
- **SD:** 1.5
- **Type:** Informative
- **Justification:** Day 10 confirmed league-level HFA = +2.48 points (p<0.001).
  Prior centered at 2.5 with SD 1.5 encodes this finding while allowing the
  posterior to update. No conference-level HFA layer — team-level deviations
  handle within-conference variation (team HFA SD = 4.85 pts confirmed Day 10).

### 1.3 Dispersion parameter r
- **Parameter:** r_negbinom
- **Distribution:** HalfNormal
- **SD:** 5.0
- **Type:** Weakly informative
- **Justification:** Day 6 confirmed VMR range 4.95–7.16 (ratio 1.447), below
  the 1.5 threshold for conference-specific dispersion. Start with a single
  league-level r. Add conference-specific r only if posterior predictive checks
  show systematic miscalibration by conference. HalfNormal constrains r > 0.

---

## 2. Conference-Level Priors

### 2.1 Conference scoring offset hyperprior
- **Parameter:** mu_conference[c] for each conference c
- **Distribution:** Normal, centered on league intercept
- **Mean:** 0.0 (offset from league baseline)
- **SD hyperprior:** HalfNormal(SD=3.0)
- **Type:** Weakly informative
- **Justification:** Day 10 confirmed conference ICC is marginal (0.02–0.05)
  but pooling still improves small-sample estimates. Hyperprior SD of 3.0
  allows realistic conference-level scoring differences while regularizing
  toward the league mean.

### 2.2 Conference-level home field deviation
- **Decision:** Not modeled as a separate layer.
- **Justification:** Day 10 found conference HFA range of 4.19 pts but team-level
  HFA SD of 4.85 pts absorbs this variation. A conference-level HFA layer is not
  justified given marginal conference ICC.

### 2.3 Conference-specific feature weights
- Features with conference_scope restrictions (see final_features.csv) receive
  zero weight outside their confirmed conference list. This is implemented as a
  conference membership indicator multiplied by the feature coefficient, not as
  a separate prior layer.

---

## 3. Team-Level Priors

### 3.1 Team attack parameter
- **Parameter:** alpha_team[t] (log-scale offensive strength)
- **Distribution:** Normal, centered on conference mean
- **Mean:** 0.0 (offset from conference baseline)
- **SD hyperprior:** HalfNormal(SD=0.4)
- **Type:** Weakly informative
- **Justification:** Day 10 team ICC for points_scored = 0.1394 — substantial,
  justifies team-level parameters. YoY r for raw scoring = 0.35–0.49, too
  unstable to use directly; prior is anchored by SP+ and EPA instead.

### 3.2 Team defense parameter
- **Parameter:** delta_team[t] (log-scale defensive strength, points allowed)
- **Distribution:** Normal, centered on conference mean
- **Mean:** 0.0 (offset from conference baseline)
- **SD hyperprior:** HalfNormal(SD=0.4)
- **Type:** Weakly informative
- **Justification:** Symmetric with attack parameter. Day 10 team ICC for
  point_differential = 0.1925 — strongest ICC finding, justifies separate
  attack and defense parameters.

### 3.3 Team home field deviation
- **Parameter:** hfa_team[t]
- **Distribution:** Normal, centered on league HFA
- **Mean:** 0.0 (deviation from league hfa_league)
- **SD hyperprior:** HalfNormal(SD=2.0)
- **Type:** Weakly informative
- **Justification:** Day 10 team HFA SD = 4.85 pts. Hyperprior SD of 2.0 on
  the log scale allows substantial team-level variation while regularizing
  sparse teams toward the league baseline.

### 3.4 SP+ prior seed
- **Parameter:** sp_weight (coefficient on SP+ rating)
- **Distribution:** Normal
- **Mean:** 0.0
- **SD:** 1.0
- **Type:** Informative
- **Justification:** YoY r = 0.7740 — strong stability. Prior decay confirmed:
  spread partial r = 0.2240 at conf game 1, does not decay monotonically
  (games 9-12 r = 0.2609). Do not aggressively down-weight as games accumulate.
  SP+ components (sp_offense, sp_defense) excluded — less stable than composite.
- **Conference variation:** American Athletic and Mid-American show strongest
  SP+ signal at game 1 (r > 0.40). Conference USA and Pac-12 have insufficient
  sample at game 1 — use league-level weight for those.

### 3.5 Recruiting prior seed
- **Parameter:** rec_weight[c] (conference-specific coefficient on recruiting_3yr_avg)
- **Distribution:** Normal
- **Mean:** 0.0
- **SD:** 0.5
- **Type:** Informative
- **Justification:** YoY r = 0.9779 — extremely stable. Prior weight is
  conference-specific:
  - Big Ten: moderate weight (rec↔sp_r = 0.7456, rec↔diff_r = 0.6601)
  - SEC: moderate weight (rec↔sp_r = 0.6730, rec↔diff_r = 0.6153)
  - All other conferences: low weight
  - Sun Belt: weight must be non-positive (rec↔diff_r = -0.2665). Recruiting
    composite does not predict positive outcomes in Sun Belt — using it as a
    positive prior signal would introduce systematic error. This is a hard
    constraint and appears as a checklist item in evaluation_checklist.md.

---

## 4. Game-Level Feature Priors

### 4.1 Close-game EPA anchor pair
- **Features:** close_game_epa_per_play, close_game_def_epa_per_play
- **Distribution:** Normal(0, 0.5) each
- **Type:** Weakly informative
- **Justification:** Joint model anchors. Spread partial r = 0.5988 and -0.6134
  at conf game 1. O/U partial r = 0.4237 and 0.4473. YoY r = 0.4331 and 0.4224.
  Signal holds across full season trajectory. SD of 0.5 allows substantial effect
  while ruling out implausible coefficients.

### 4.2 Pregame ELO
- **Feature:** pregame_elo
- **Distribution:** Normal(0, 0.3)
- **Type:** Weakly informative
- **Justification:** Spread partial r = 0.1702; holds at conf game 1 (r = 0.1870).
  YoY r = 0.8452 — highly stable game-level predictor. Spread signal only; O/U
  signal absent.

### 4.3 ELO/SP+ divergence
- **Feature:** elo_sp_divergence
- **Distribution:** Normal(0, 0.2)
- **Type:** Weakly informative
- **Justification:** Spread r = 0.1650 after SP+ controlled. Smaller SD than
  pregame_elo because this is an interaction-style feature capturing disagreement
  between two rating systems. Computed in notebook; add to dbt only after model
  confirms value. (Ambiguity 5 resolved: include.)

### 4.4 Rolling momentum features
- **Features:** last3_win_pct, last3_off_epa_avg, last3_def_epa_avg,
  last3_points_scored_avg, last3_points_allowed_avg
- **Distribution:** Normal(0, 0.3) each
- **Type:** Weakly informative
- **Justification:** Conference-specific spread signal from conf game 2. Null at
  conf game 1 — handled by null_handling = impute_season_prior (replace with
  season-to-date average, consistent with Approach A early-season null handling
  confirmed as locked decision).
- **Conference scope:** Applied only within confirmed conference lists per
  final_features.csv. last3_win_pct applied across all conferences.

### 4.5 Environmental features (threshold-activated)
- **Features:** away_elevation_delta_ft, away_travel_distance_mi, away_tz_delta_hrs,
  wind_chill
- **Distribution:** Normal(0, 0.3) for elevation/travel/timezone;
  Normal(0, 0.2) for wind_chill
- **Type:** Weakly informative
- **Justification:** All are threshold-activated — signal only emerges above
  specific thresholds. Modeled as indicator×magnitude interaction:
  - away_elevation_delta_ft: active when delta >= 2000 ft (YoY r = 0.8255)
  - away_travel_distance_mi: active when distance >= 1500 mi (YoY r = 0.6562)
  - away_tz_delta_hrs: active when abs(delta) >= 2 hr (YoY r = 0.6710);
    direction negative
  - wind_chill: active when <= 40°F and NOT is_dome; O/U signal only
  When threshold not met, feature value = 0 (null_handling = zero).

### 4.6 Style/tempo delta features
- **Features:** rush_rate_std_downs_delta, rush_rate_pass_downs_delta,
  off_pts_per_opportunity_delta, def_pts_per_opportunity_allowed_delta,
  off_success_rate_pass_delta, def_success_rate_pass_allowed_delta,
  off_epa_pass_delta, def_epa_pass_allowed_delta
- **Distribution:** Normal(0, 0.3) each
- **Type:** Weakly informative
- **Justification:** Spread signal confirmed in Day 15 at game level. Rush tendency
  (rush_rate_std_downs_delta) is the most consistent — holds across all season
  buckets including game 1 (r = 0.2965–0.3628). Pass efficiency deltas confirmed.
  YoY stability insufficient for prior seeding (best r = 0.4890 for
  rush_rate_std_downs); treated as game-level predictors only.

### 4.7 Sack-rate mismatch features (moneyline variance)
- **Features:** off_sack_rate_allowed_delta, def_sack_rate_delta
- **Distribution:** Normal(0, 0.2)
- **Type:** Weakly informative
- **Justification:** Moneyline variance candidates. Abs residual variance partial
  r = ±0.0919. Smaller SD than spread features — weaker signal, more regularization
  needed. These affect the dispersion component of the model, not the mean.

### 4.8 Style archetype matchup features
- **Features:** offense_archetype_matchup, defense_archetype_matchup,
  home_off_vs_away_def_matchup, away_off_vs_home_def_matchup
- **Distribution:** Normal(0, 0.3) each
- **Type:** Weakly informative
- **Justification:** O/U signal confirmed — strongest EDA 10 finding (eta² up to
  0.39). Weak secondary spread signal. Not valid for moneyline variance. Not stable
  for prior seeding (offense retention 0.26–0.35 YoY, defense 0.25–0.40 YoY).
  (Ambiguity 2 resolved: include with deployable pregame version requirement.)
- **CONDITION:** Deployable pregame or rolling version must be tested before
  September 24, 2026 production launch. If no pregame version clears signal tests,
  these four features are dropped at that stage.

### 4.9 Days since last game (bye week)
- **Feature:** days_since_last_game
- **Distribution:** Normal(0, 0.2)
- **Type:** Weakly informative
- **Justification:** Bye week signal (>= 12 days) in American Athletic and Big 12
  only. Threshold-activated; zero outside confirmed conferences.
- **Conference scope:** American Athletic, Big 12 only.

### 4.10 Rolling EPA conference-specific features
- **Features:** last3_off_epa_avg, last3_def_epa_avg (repeated here for clarity
  on conference scope)
- **Distribution:** Normal(0, 0.3) each
- **Conference scope:**
  - last3_off_epa_avg: ACC, Mid-American, SEC
  - last3_def_epa_avg: American Athletic, Big Ten, Conference USA,
    Mid-American, Pac-12, Sun Belt
- **Justification:** Conference lists differ between offense and defense —
  do not consolidate. (Ambiguity 3 resolved: include with explicit lists.)

---

## Summary: Parameter Count

| Layer          | Parameters                                      | Count |
|----------------|-------------------------------------------------|-------|
| League         | mu_league, hfa_league, r_negbinom               | 3     |
| Conference     | mu_conference[c] × 10 conferences               | 10    |
| Team           | alpha_team[t], delta_team[t], hfa_team[t],      |       |
|                | sp_weight, rec_weight[c]                        | 3N+1+10|
| Game-level     | 23 feature coefficients (see final_features.csv)| 23    |

N = number of teams in training data (approximately 130 FBS teams × 3 seasons).

---

## Day 20 Instructions

1. Translate every prior above into a `pm.` distribution call.
2. Every parameter must have a comment citing this document and the EDA finding.
3. Do not invent priors. If a parameter is not listed here, stop and flag it.
4. Conference-specific feature weights are implemented as coefficient × conference
   indicator — do not create separate conference-level coefficient distributions
   for game-level features unless the prior specification explicitly calls for it.
5. Sun Belt recruiting weight must be non-positive. Implement as a constraint or
   as a separate non-positive prior for that conference's recruiting coefficient.
"""

prior_spec_path = os.path.join(MAIN_ART, "prior_specification_draft.md")
with open(prior_spec_path, "w") as f:
    f.write(prior_spec_md)

print(f"✓ prior_specification_draft.md written → {prior_spec_path}")
print(f"  Size: {os.path.getsize(prior_spec_path):,} bytes")

# ── Final artifact inventory ─────────────────────────────────────────────────
print("\n=== Day 19 artifact inventory ===")
artifacts_to_check = [
    os.path.join(MAIN_ART, "master_verdict.csv"),
    os.path.join(MAIN_ART, "final_features.csv"),
    os.path.join(MAIN_ART, "ambiguity_resolution.md"),
    os.path.join(MAIN_ART, "prior_specification_draft.md"),
]
for p in artifacts_to_check:
    exists = os.path.exists(p)
    size   = os.path.getsize(p) if exists else 0
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {p}  ({size:,} bytes)")

✓ prior_specification_draft.md written → /Users/kevinjohnson/cfb-analytics/artifacts/prior_specification_draft.md
  Size: 12,599 bytes

=== Day 19 artifact inventory ===
  ✓  /Users/kevinjohnson/cfb-analytics/artifacts/master_verdict.csv  (21,198 bytes)
  ✓  /Users/kevinjohnson/cfb-analytics/artifacts/final_features.csv  (10,007 bytes)
  ✓  /Users/kevinjohnson/cfb-analytics/artifacts/ambiguity_resolution.md  (4,478 bytes)
  ✓  /Users/kevinjohnson/cfb-analytics/artifacts/prior_specification_draft.md  (12,599 bytes)


In [11]:
# ── Close DB connection ──────────────────────────────────────────────────────
conn.close()
print("DB connection closed.\n")

# ── Day 19 completion summary ────────────────────────────────────────────────
print("=" * 65)
print("DAY 19 — EDA FINALIZATION COMPLETE")
print("=" * 65)

print("""
OUTPUTS PRODUCED
────────────────
1. master_verdict.csv (93 rows)
   One row per candidate feature. All EDA days consolidated.
   include: 23  |  exclude: 70
   Roles: anchor=3, prior_seed=2, conference_specific=8,
          supporting=10, excluded=70 (includes 31 untested Day 14)
   Ambiguity flags: 10 rows (all 5 ambiguities represented)

2. final_features.csv (23 rows)
   Filtered to include only. Every row has complete prior spec.
   model_layer: game_level=21, team=2
   null_handling: impute_season_prior=7, zero=5, not_applicable=11

3. ambiguity_resolution.md
   5 binding decisions — all resolved as INCLUDE (with conditions
   noted for archetypes and elo_sp_divergence).

4. prior_specification_draft.md
   Every model parameter specified with distribution family, mean,
   SD, and EDA justification. Day 20 translates this into PyMC code
   without inventing any priors.

AMBIGUITY RESOLUTIONS (binding)
────────────────────────────────
1. close_game_play_count_delta     → INCLUDE (6 conferences)
2. Style archetype matchup (×4)    → INCLUDE (pregame version required)
3. last3 rolling EPA conf lists    → INCLUDE with explicit lists
4. rush_rate prior seeds           → EXCLUDE as prior seeds
                                     (retained as game-level predictors)
5. elo_sp_divergence               → INCLUDE (notebook-computed)

WHAT DAY 20 MUST DO
────────────────────
Read prior_specification_draft.md before writing any PyMC code.
Every pm. distribution call must cite the EDA finding that motivated it.
Do not invent priors. If a parameter is not in the spec, stop and flag it.
Sun Belt recruiting weight must be non-positive — hard constraint.
""")
print("=" * 65)

DB connection closed.

DAY 19 — EDA FINALIZATION COMPLETE

OUTPUTS PRODUCED
────────────────
1. master_verdict.csv (93 rows)
   One row per candidate feature. All EDA days consolidated.
   include: 23  |  exclude: 70
   Roles: anchor=3, prior_seed=2, conference_specific=8,
          supporting=10, excluded=70 (includes 31 untested Day 14)
   Ambiguity flags: 10 rows (all 5 ambiguities represented)

2. final_features.csv (23 rows)
   Filtered to include only. Every row has complete prior spec.
   model_layer: game_level=21, team=2
   null_handling: impute_season_prior=7, zero=5, not_applicable=11

3. ambiguity_resolution.md
   5 binding decisions — all resolved as INCLUDE (with conditions
   noted for archetypes and elo_sp_divergence).

4. prior_specification_draft.md
   Every model parameter specified with distribution family, mean,
   SD, and EDA justification. Day 20 translates this into PyMC code
   without inventing any priors.

AMBIGUITY RESOLUTIONS (binding)
─────────────────────